In [ ]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials import buildings as bld
from imagematerials import vehicles as vhc
from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)


In [ ]:
def get_vema_prep():
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        climate_policy_scenario_dir = base_dir / 'SSP2'
        circular_economy_scenario_dirs = {"slow": base_dir / 'circular_economy_scenarios' / 'slow'}

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
            circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
            prep_data = vhc.preprocess(base_dir, climate_policy_config, circular_economy_config)

        export_to_netcdf(prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph

    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    new_prep_data["shares"] = None
    sec_vhc = Sector("vhc", new_prep_data)
    return sec_vhc

In [ ]:
def get_buildings_prep():
    circularity_scenario = "base"
    climate_scenario = 'IMAGE_CircoMod/SSP2'
    climate_policy_scenario_dir = Path("../data/raw") / climate_scenario
    scenario_path = Path("../data/raw") / 'circular_economy_scenarios' / circularity_scenario
    circular_economy_scenario_dirs = {circularity_scenario: scenario_path}
    climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
    circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir, climate_policy_config, circular_economy_config)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [ ]:
sectors = [get_vema_prep(), get_buildings_prep()]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [ ]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [ ]:
model.combi["summed_inflow_materials"][2020]